# Checkpoint 2 - Submission B
Pipeline hierárquica:
1. Classificador binário: Human vs AI
2. Classificador multiclasse para textos AI: Google / Meta / Mistral / OpenAI

Abordagens:
- TF-IDF word n-grams
- TF-IDF char n-grams
- Logistic Regression (binário)
- Linear SVM (multiclasse AI)

Objetivo:
- melhorar robustez em textos curtos
- aproximar treino do formato de validação externa

## Imports

In [13]:
import os
import re
import json
import joblib
import numpy as np
import pandas as pd

from collections import Counter

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

## Caminhos

In [14]:
DATA_DIR = "../data"
MODELS_DIR = "../models"

TRAIN_PATH = os.path.join(DATA_DIR, "dataset_treino.csv")
VAL_PATH = os.path.join(DATA_DIR, "dataset_validacao.csv")
TEST_PATH = os.path.join(DATA_DIR, "dataset_teste.csv")

os.makedirs(MODELS_DIR, exist_ok=True)

## Loading

In [15]:
df_train = pd.read_csv(TRAIN_PATH)
df_val = pd.read_csv(VAL_PATH)
df_test = pd.read_csv(TEST_PATH)

print("Train:", df_train.shape)
print("Val:", df_val.shape)
print("Test:", df_test.shape)

display(df_train.head())

Train: (10880, 3)
Val: (2332, 3)
Test: (2332, 3)


,Text,source_name,source_code
0,Shares will be acquired in accordance with sec...,human,0
1,lmao just saw a funny meme rn,meta,3
2,As the hit TV show Downton Abbey continues to ...,meta,3
3,The building will house product development an...,human,0
4,Nigella Lawson has finally broken her silence ...,human,0


In [16]:
print("Train columns:", df_train.columns.tolist())
print("Val columns:", df_val.columns.tolist())
print("Test columns:", df_test.columns.tolist())

Train columns: ['Text', 'source_name', 'source_code']
Val columns: ['Text', 'source_name', 'source_code']
Test columns: ['Text', 'source_name', 'source_code']


In [ ]:
TEXT_COL = "Text"
LABEL_COL = "source_name"
ID_COL = "source_code" if "source_code" in df_test.columns else None

## Pré-Processamento

In [18]:
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

### LIMITAR TAMANHO

In [19]:
def clip_text_for_external_style(text, min_chars=80, max_chars=140):
    text = clean_text(text)
    if len(text) < min_chars:
        return None
    return text[:max_chars]

In [20]:
for df in [df_train, df_val, df_test]:
    df[TEXT_COL] = df[TEXT_COL].astype(str).apply(clean_text)

df_train["text_model"] = df_train[TEXT_COL].apply(clip_text_for_external_style)
df_val["text_model"] = df_val[TEXT_COL].apply(clip_text_for_external_style)
df_test["text_model"] = df_test[TEXT_COL].apply(clip_text_for_external_style)

df_train = df_train[df_train["text_model"].notna()].copy()
df_val = df_val[df_val["text_model"].notna()].copy()
df_test = df_test[df_test["text_model"].notna()].copy()

print(df_train.shape, df_val.shape, df_test.shape)

KeyError: 'text'